

**Objetivo**: Aplicar técnicas avanzadas de manipulación y análisis de datos con pandas sobre un conjunto real de datos de contenido de Netflix, reforzando buenas prácticas y métodos eficientes sin recurrir a `groupby`, `merge`, `pivot`, ni `join`.



**Dataset**:

Trabajaremos con el archivo `netflix_titles.csv`, que contiene información sobre los títulos disponibles en la plataforma Netflix hasta el año 2021.

| Variable       | Clase     | Descripción                                                                 |
|----------------|-----------|------------------------------------------------------------------------------|
| show_id        | caracter  | Identificador único del título en el catálogo de Netflix.                   |
| type           | caracter  | Tipo de contenido: 'Movie' o 'TV Show'.                                     |
| title          | caracter  | Título del contenido.                                                       |
| director       | caracter  | Nombre del director (puede ser nulo).                                       |
| cast           | caracter  | Lista de actores principales (puede ser nulo).                              |
| country        | caracter  | País o países donde se produjo el contenido.                                |
| date_added     | fecha     | Fecha en la que el título fue agregado al catálogo de Netflix.              |
| release_year   | entero    | Año de lanzamiento original del título.                                     |
| rating         | caracter  | Clasificación por edad (por ejemplo: 'PG-13', 'TV-MA').                      |
| duration       | caracter  | Duración del contenido (minutos o número de temporadas para series).        |
| listed_in      | caracter  | Categorías o géneros en los que está clasificado el contenido.              |
| description    | caracter  | Breve sinopsis del contenido.                                               |




In [ ]:
import pandas as pd

# Cargar datos
df = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...



### Parte 1: Limpieza y preparación

1. Revisar y describir el dataset:

   * ¿Cuántas filas y columnas tiene?
   * ¿Qué tipos de datos hay?
   * ¿Cuántos valores nulos hay por columna?

2. Transformar la columna `date_added` a tipo fecha.

3. Crear columnas auxiliares con `assign`:

   * Año (`year_added`)
   * Mes (`month_added`)



In [ ]:
# 1. Revisar y describir el dataset
print(df.shape)
print("-" * 50)
print(df.isnull().sum())
print("-" * 50)
print(df.dtypes)
print("-" * 50)

# 2. Transformar columna date_added a tipo fecha
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
print(df['date_added'].dtype)

print("-" * 50)

# 3. Crear columnas auxiliares con assign
df = df.assign(
    year_added=df['date_added'].dt.year,
    month_added=df['date_added'].dt.month
)
print(df[['date_added', 'year_added', 'month_added']].head())

(8807, 14)
--------------------------------------------------
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
year_added        10
month_added       10
dtype: int64
--------------------------------------------------
show_id                 object
type                    object
title                   object
director                object
cast                    object
country                 object
date_added      datetime64[ns]
release_year             int64
rating                  object
duration                object
listed_in               object
description             object
year_added             float64
month_added            float64
dtype: object
--------------------------------------------------
datetime64[ns]
--------------------------------------------------
  date_added  y

### Parte 1: Limpieza y preparación

1. Revisar y describir el dataset:

¿Cuántas filas y columnas tiene?
Tiene 8807 filas y 12 columnas.

¿Qué tipos de datos hay?
Hay datos de tipo INT64 y Object.

¿Cuántos valores nulos hay por columna?
En show_id = 0
type = 0,
title = 0,
director = 2634,
cast = 825,
country = 831,
date_added = 10,
release_year = 0,
rating = 4,
duration = 3,
listed_in = 0,
description = 0.



## Parte 2: Técnicas avanzadas de pandas

4. Utilizar `.loc` para seleccionar películas (`type == 'Movie'`) que fueron agregadas después del año 2018.

5. Utilizar `str.contains()` y `str.extract()`:

   * Filtrar títulos que contienen la palabra 'love' (sin distinguir mayúsculas/minúsculas).
   * Extraer la duración en minutos para las películas desde la columna `duration`.

6. Aplicar `explode()` sobre la columna `listed_in` para obtener una fila por cada género.

7. Obtener un top 10 de géneros más frecuentes utilizando `value_counts()`.

8. Aplicar `where()` y `mask()` para marcar las películas de más de 120 minutos como contenido largo en una nueva columna.

9. Utilizar `.loc` para filtrar películas que cumplen con:

   * Más de 100 minutos de duración.
   * Rating igual a `'R'`.
   * País igual a `'United States'`.

10. Utilizar `.style` para formatear visualmente el top 10 de películas más largas.

In [ ]:
import numpy as np

# 4. Filtrar con .loc (type == 'Movie' y año > 2018)
peliculas_post_2018 = df.loc[(df['type'] == 'Movie') & (df['year_added'] > 2018)]
print("Resultado Pregunta 4:")
print(peliculas_post_2018.head())
print("-" * 50)


#  5. Utilizar str.contains() y str.extract()
# Filtrar títulos con 'love'
titulos_love = df[df['title'].str.contains('love', case=False, na=False)]
print("Resultado Pregunta 5 (Títulos con 'love'):")
print(titulos_love['title'].head())

# Extraer duración en minutos
df['minutes'] = df['duration'].str.extract(r'(\d+)').astype(float)
print("\nResultado Pregunta 5 (Extracción de minutos):")
print(df[['duration', 'minutes']].head())
print("-" * 50)


# 6. Aplicar explode() sobre la columna listed_in
df['genres_list'] = df['listed_in'].str.split(', ')
df_exploded = df.explode('genres_list')
print("Resultado Pregunta 6:")
print(df_exploded[['title', 'genres_list']].head())
print("-" * 50)


# 7. Obtener top 10 de géneros más frecuentes con value_counts()
top_10_generos = df_exploded['genres_list'].value_counts().head(10)
print("Resultado Pregunta 7:")
print(top_10_generos)
print("-" * 50)


# 8. Aplicar where() y mask()
df['contenido_largo'] = np.nan
df['contenido_largo'] = df['contenido_largo'].mask((df['type'] == 'Movie') & (df['minutes'] > 120), 'Contenido Largo')
print("Resultado Pregunta 8:")
print(df.loc[df['minutes'] > 120, ['title', 'duration', 'contenido_largo']].head())
print("-" * 50)


# 9. Utilizar .loc para filtrar películas bajo 3 condiciones
peliculas_filtradas = df.loc[
    (df['type'] == 'Movie') &
    (df['minutes'] > 100) &
    (df['rating'] == 'R') &
    (df['country'] == 'United States')
]
print("Resultado Pregunta 9:")
print(peliculas_filtradas.head())
print("-" * 50)


# 10. Utilizar .style para formatear visualmente el top 10 de películas más largas
# Nota: Este debe ir al final de la celda para que Colab renderice la tabla interactiva
print("L a tabla estilizada a continuación):")
top_10_largas = df[df['type'] == 'Movie'].nlargest(10, 'minutes')
top_10_estilizado = top_10_largas[['title', 'minutes']].style.bar(subset=['minutes'], color='#d65f5f')
top_10_estilizado

Resultado Pregunta 4:
   show_id   type                             title  \
0       s1  Movie              Dick Johnson Is Dead   
6       s7  Movie  My Little Pony: A New Generation   
7       s8  Movie                           Sankofa   
9      s10  Movie                      The Starling   
12     s13  Movie                      Je Suis Karl   

                         director  \
0                 Kirsten Johnson   
6   Robert Cullen, José Luis Ucha   
7                    Haile Gerima   
9                  Theodore Melfi   
12            Christian Schwochow   

                                                 cast  \
0                                                 NaN   
6   Vanessa Hudgens, Kimiko Glenn, James Marsden, ...   
7   Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...   
9   Melissa McCarthy, Chris O'Dowd, Kevin Kline, T...   
12  Luna Wedler, Jannis Niewöhner, Milan Peschel, ...   

                                              country date_added  \
0            

,title,minutes
4253,Black Mirror: Bandersnatch,312.000000
717,Headspace: Unwind Your Mind,273.000000
2491,The School of Mischief,253.000000
2487,No Longer kids,237.000000
2484,Lock Your Girls In,233.000000
2488,Raya and Sakina,230.000000
166,Once Upon a Time in America,229.000000
7932,Sangam,228.000000
1019,Lagaan,224.000000
4573,Jodhaa Akbar,214.000000




### Pregunta Desafío

11. ¿Cuáles son las combinaciones más frecuentes de género y rating en el dataset?
    (Sugerencia: utilizar `value_counts` con `subset=["genre", "rating"]` después de aplicar `explode()`).



### Bonus: Análisis de duplicados y limpieza

12. ¿Existen películas con el mismo nombre (`title`) pero con distinto año de lanzamiento (`release_year`)?
13. ¿Cuántos títulos únicos hay en total en la columna `title`?





In [ ]:
# 11. ¿Cuáles son las combinaciones más frecuentes de género y rating?
# Nota: La sugerencia pide usar subset=['genre', 'rating'], pero nuestra columna del explode se llama 'genres_list'
combinaciones_frecuentes = df_exploded.value_counts(subset=['genres_list', 'rating'])
print("Combinaciones más frecuentes:")
print(combinaciones_frecuentes.head(10))
print("-" * 50)


# 12. ¿Existen películas con el mismo nombre pero con distinto año de lanzamiento?
# Filtramos solo las películas
peliculas = df[df['type'] == 'Movie']

# Buscamos títulos duplicados en el dataframe de películas
titulos_duplicados = peliculas[peliculas.duplicated(subset=['title'], keep=False)]

# Agrupamos por título para ver si tienen años distintos (contando cuántos años únicos hay por título)
# Como no podemos usar groupby, usamos otra estrategia con .loc o filtrando combinaciones únicas primero
combinaciones_unicas = peliculas[['title', 'release_year']].drop_duplicates()
nombres_repetidos_con_distinto_ano = combinaciones_unicas[combinaciones_unicas.duplicated(subset=['title'], keep=False)]

print("Películas con mismo nombre pero distinto año:")
print(nombres_repetidos_con_distinto_ano.sort_values(by='title').head(10))
print("-" * 50)


# 13. ¿Cuántos títulos únicos hay en total en la columna title?
total_titulos_unicos = df['title'].nunique()
print(f"Total de títulos únicos: {total_titulos_unicos}")
print("-" * 50)

Combinaciones más frecuentes:
genres_list             rating
International Movies    TV-MA     1130
                        TV-14     1065
Dramas                  TV-MA      830
International TV Shows  TV-MA      714
Dramas                  TV-14      693
International TV Shows  TV-14      472
Comedies                TV-14      465
TV Dramas               TV-MA      434
Comedies                TV-MA      431
Dramas                  R          375
Name: count, dtype: int64
--------------------------------------------------
Películas con mismo nombre pero distinto año:
Empty DataFrame
Columns: [title, release_year]
Index: []
--------------------------------------------------
Total de títulos únicos: 8807
--------------------------------------------------
